# 02 — Feature Engineering

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

DATA_DIR = '/content/drive/MyDrive/BSE405_SoilMoisture/data'

df = pd.read_csv(f'{DATA_DIR}/updated_data.csv', parse_dates=['time'])
print(f"Original shape: {df.shape}")
df.head()

## Temporal Features

In [ ]:
df['day_of_year'] = df['time'].dt.dayofyear

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['season'] = df['time'].dt.month.apply(get_season)

print(df[['time', 'day_of_year', 'season']].head(10))
print(f"\nSeason counts:\n{df['season'].value_counts()}")

## Soil Texture Ratios and Interaction Terms

In [ ]:
df['clay_sand_ratio'] = df['clay_content'] / df['sand_content'].replace(0, np.nan)
df['clay_silt_ratio'] = df['clay_content'] / df['silt_content'].replace(0, np.nan)
df['clay_x_sm_aux'] = df['clay_content'] * df['sm_aux']

print(df[['clay_sand_ratio', 'clay_silt_ratio', 'clay_x_sm_aux']].describe())

## Spatial Split Columns

In [ ]:
# West/East split (~10.5 degrees longitude, approximate historical border)
df['region'] = np.where(df['longitude'] < 10.5, 'West', 'East')
print(f"Region counts:\n{df['region'].value_counts()}")

# Spatial block CV — divide into 6 blocks (2 lat bands x 3 lon bands)
lat_min, lat_max = df['latitude'].min(), df['latitude'].max()
lon_min, lon_max = df['longitude'].min(), df['longitude'].max()

lat_bins = np.linspace(lat_min, lat_max + 0.01, 3)  # 2 lat bands
lon_bins = np.linspace(lon_min, lon_max + 0.01, 4)  # 3 lon bands

df['lat_band'] = pd.cut(df['latitude'], bins=lat_bins, labels=['S', 'N'])
df['lon_band'] = pd.cut(df['longitude'], bins=lon_bins, labels=['W', 'C', 'E'])
df['spatial_block'] = df['lat_band'].astype(str) + '_' + df['lon_band'].astype(str)

print(f"\nSpatial block counts:\n{df['spatial_block'].value_counts()}")

## Handle Missing Values

In [ ]:
print(f"Missing values before cleanup:\n{df.isnull().sum()}")
print(f"\nTotal rows before: {len(df)}")

df_clean = df.dropna(subset=['sm_aux', 'sm_tgt', 'clay_sand_ratio', 'clay_silt_ratio'])
print(f"Total rows after: {len(df_clean)}")
print(f"Rows dropped: {len(df) - len(df_clean)}")

## Export Processed Data

In [ ]:
df_clean = df_clean.drop(columns=['lat_band', 'lon_band'])

print(f"Final shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")

df_clean.to_csv(f'{DATA_DIR}/processed_data.csv', index=False)
print(f"\nSaved to {DATA_DIR}/processed_data.csv")

## Verify Export

In [ ]:
verify = pd.read_csv(f'{DATA_DIR}/processed_data.csv')
print(f"Verified shape: {verify.shape}")
verify.head()